# 2. Demand Estimation

## This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

First, import the necessary packages and examine the data

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [5]:
df = pd.read_csv("air_fryers_clean_brand_year.csv")

display(df.head())
print(df.shape)
print(df.columns)

,category,year,brand,purchase_count,product_count,avg_price,avg_rating,compact_share,dual_basket_share,oven_style_share,rotisserie_share,window_share,market_purchases,brand_share,log_brand_share
0,air_fryers,2019,chefman,1146,10,72.963695,4.434119,1.000000,0.0,0.780977,0.243455,0.184119,15076,0.076015,-2.576826
1,air_fryers,2019,cosori,11,2,159.990000,4.581818,1.000000,0.0,0.090909,0.090909,0.000000,15076,0.000730,-7.222964
2,air_fryers,2019,cuisinart,1616,22,229.465274,4.481312,0.993812,0.0,0.889851,0.000000,0.000000,15076,0.107190,-2.233150
3,air_fryers,2019,dash,3011,19,55.176333,4.390767,1.000000,0.0,0.973431,0.000000,0.000000,15076,0.199721,-1.610832
4,air_fryers,2019,gowise usa,4405,45,83.575551,4.552259,0.999773,0.0,0.129398,0.128490,0.000000,15076,0.292186,-1.230364


(50, 15)
Index(['category', 'year', 'brand', 'purchase_count', 'product_count',
       'avg_price', 'avg_rating', 'compact_share', 'dual_basket_share',
       'oven_style_share', 'rotisserie_share', 'window_share',
       'market_purchases', 'brand_share', 'log_brand_share'],
      dtype='str')


Then, make the linear regression model

In [ ]:
# Identify air fryer feature columns
feature_cols = [
    "compact_share",
    "dual_basket_share",
    "oven_style_share",
    "rotisserie_share",
    "window_share"
]

# Create brand & year dummy variables
brand_dummies = pd.get_dummies(df["brand"], prefix="brand", drop_first=True, dtype=int)
year_dummies = pd.get_dummies(df["year"].astype(str), prefix="year", drop_first=True, dtype=int)

# Concatenate variables to make X matrix of input variables (independent variables)
X = pd.concat(
    [
        df[["avg_price", "avg_rating"] + feature_cols],
        brand_dummies,
        year_dummies
    ],
    axis=1
)

# Select the y outcome variable that we are trying to predict (dependent variable)
y = df["log_brand_share"]

# Make and fit the model
model = LinearRegression()
model.fit(X, y)

# Use the model to make predictions
predicted_log_share = model.predict(X)

# Make a coeffecients table 
coef_table = pd.DataFrame({
    "variable": X.columns,
    "coefficient": model.coef_
})

display(coef_table)

,variable,coefficient
0,avg_price,-0.037668
1,avg_rating,0.287517
2,compact_share,9.815304
3,dual_basket_share,-9.509686
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
6,window_share,12.880298
7,brand_cosori,2.551946
8,brand_cuisinart,6.422436
9,brand_dash,0.176655


1. What is the estimated price coefficient, $\hat{\beta}_{price}$?

In [ ]:
price_coef = coef_table.loc[
    coef_table["variable"] == "avg_price",
    "coefficient"
].iloc[0]

print("The estimated price coefficient is about", round(price_coef, 4))

The estimated price coefficient is -0.0377


2. Is it negative? Why is that important?

Yes, the estimated price coefficient is negative. This is important because demand should decrease when price increases, holding the other variables constant. A negative price coefficient means the model follows the basic economic logic of demand.

3. Which product features are associated with higher demand?

In [ ]:
feature_coefs = coef_table[
    coef_table["variable"].isin(feature_cols)
].sort_values("coefficient", ascending=False)

display(feature_coefs)

,variable,coefficient
6,window_share,12.880298
2,compact_share,9.815304
4,oven_style_share,1.941774
5,rotisserie_share,-5.674054
3,dual_basket_share,-9.509686


The features associated with higher demand are those with positive coefficients: `window_share`, `compact_share`, and `oven_style_share`. This means those features predict higher demand, holding other factors constant.

4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.

In [ ]:
brand_coefs = coef_table[
    coef_table["variable"].str.startswith("brand_")
].sort_values("coefficient", ascending=False)

display(brand_coefs.head())

,variable,coefficient
8,brand_cuisinart,6.422436
12,brand_ninja,5.838705
11,brand_instant_pot,4.626260
10,brand_gowise usa,3.938996
14,brand_oster,3.928074


The largest brand dummy coefficients are `brand_cuisinart`, `brand_ninja`, and `brand_instant_pot`. These brands have higher predicted demand than the reference brand, holding other factors constant.

5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.

In [ ]:
year_coefs = coef_table[
    coef_table["variable"].str.startswith("year_")
].sort_values("coefficient", ascending=False)

display(year_coefs.head())

,variable,coefficient
16,year_2020,0.119071
17,year_2021,0.041900
19,year_2023,-0.003307
18,year_2022,-0.098860


The largest year dummy coefficients are `year_2020`, `year_2021`, and `year_2023`. These indicate higher predicted demand relative to the reference year, holding other factors constant.

6. What is the model's $R^2$?

In [ ]:
r2 = r2_score(y, predicted_log_share)

print("r squared:", round(r2, 4))

r squared: 0.7635


This means that the model explains about 76.4% of the variation in log brand share. Overall, this suggests the model fits the air fryer brand-year data fairly well.